<a href="https://colab.research.google.com/github/korkutanapa/ANOMALY_DETECTION_TDA_YAHOO_DATASET/blob/main/POLYMER_DESIGN_SIMULATOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PREDICTION APP FOR MATERIAL DESIGN

In [1]:
# @title RUN ONCE BEFORE PREDICTION
import numpy as np
import pandas as pd
import joblib
import re

# ============================================================
# PURPOSE
# - Your stage-2 model expects engineered columns like:
#   Break Strength (Mpa)_a__pred_over_density
#   Density_a__pred_x_Break Strength (Mpa)_a__pred
#   ...
# - This code:
#   1) asks for the 22 base inputs
#   2) predicts the 6 "__pred" values (stage-1 models)
#   3) creates ALL engineered features needed for stage-2
#   4) aligns feature columns to exactly what stage-2 models expect
#   5) predicts the last 3 outputs
# ============================================================

# -----------------------------
# Load models (update paths if needed)
# -----------------------------
MODEL_PATHS = {
    # ---- Stage-1 outputs (predicted first) ----
    "Density_a__pred": "/content/best_Density_a.joblib",
    "Tensile Strength (Mpa)_a__pred": "/content/best_Tensile_Strength_(Mpa)_a.joblib",
    "Elongation at Yield (%)_a__pred": "/content/best_Elongation_at_Yield_(%)_a.joblib",
    "Break Strength (Mpa)_a__pred": "/content/best_Break_Strength_(Mpa)_a.joblib",
    "Izod İmpact_a__pred": "/content/best_Izod_İmpact_a.joblib",
    "Flexural Strength (Mpa)_a__pred": "/content/best_Flexural_Strength_(Mpa)_a.joblib",

    # ---- Stage-2 outputs (predicted second) ----
    "Elastic Modulus (Mpa)_a": "/content/best_Elastic_Modulus_(Mpa)_a.joblib",
    "Elongation at Break (%)_a": "/content/best_Elongation_at_Break_(%)_a.joblib",
    "Flexural Modulus (Mpa)_a": "/content/best_Flexural_Modulus_(Mpa)_a.joblib",
}

models = {k: joblib.load(v) for k, v in MODEL_PATHS.items()}

# -----------------------------
# Base inputs (22)
# -----------------------------
input_columns = [
    '9711900_a', '9710700_a', '9710702_a', '9710190_a',
    '9710703_a', '9712055_a', '9710293_a', '9710278_a',
    '9710709_a', '9710475_a', '9710275_a',
    'Cal_a', 'Talc_a', '9801084_a', '9710146_a',
    'PEA0330648_a', 'PP070G2M_a', 'BE961MO_a',
    '9710156_a', '9710169_a',
    'Beyaz_a', 'Gri_a',
]

stage1_pred_cols = [
    "Density_a__pred",
    "Tensile Strength (Mpa)_a__pred",
    "Elongation at Yield (%)_a__pred",
    "Break Strength (Mpa)_a__pred",
    "Izod İmpact_a__pred",
    "Flexural Strength (Mpa)_a__pred",
]

stage2_targets = [
    "Elastic Modulus (Mpa)_a",
    "Elongation at Break (%)_a",
    "Flexural Modulus (Mpa)_a",
]

# -----------------------------
# Ask user inputs
# -----------------------------
def ask_inputs() -> pd.DataFrame:
    print("\nEnter the 22 input values (numbers). For Beyaz_a/Gri_a use 0 or 1.\n")
    row = {}
    for col in input_columns:
        while True:
            val = input(f"{col}: ").strip()
            try:
                row[col] = float(val)
                break
            except ValueError:
                print("  ❌ Please enter a numeric value.")
    return pd.DataFrame([row], columns=input_columns)

# -----------------------------
# Feature engineering for Stage-2
#   (creates the missing *_x_* and *_over_density etc.)
# -----------------------------
def build_stage2_features(df_base: pd.DataFrame) -> pd.DataFrame:
    """
    df_base: one-row DF containing input_columns + stage1_pred_cols
    returns: engineered DF containing base + engineered features
    """
    eps = 1e-12
    df = df_base.copy().apply(pd.to_numeric, errors="coerce")

    # composition features from original inputs (optional but harmless)
    comp_cols = [c for c in input_columns]  # treat all 22 as composition-like
    df["comp_sum"] = df[comp_cols].sum(axis=1)
    df["comp_nonzero_count"] = (df[comp_cols].fillna(0).abs() > 0).sum(axis=1)
    df["comp_mean_nonzero"] = df["comp_sum"] / (df["comp_nonzero_count"] + eps)
    df["comp_std"] = df[comp_cols].std(axis=1)
    df["comp_cv"] = df["comp_std"] / (df[comp_cols].mean(axis=1) + eps)
    df["comp_max"] = df[comp_cols].max(axis=1)
    df["comp_max_share"] = df["comp_max"] / (df["comp_sum"] + eps)

    # filler features
    df["filler_sum"] = df["Cal_a"].fillna(0) + df["Talc_a"].fillna(0)
    df["talc_cal_ratio"] = df["Talc_a"] / (df["Cal_a"] + eps)
    df["filler_share"] = df["filler_sum"] / (df["comp_sum"] + eps)

    # color flags
    df["is_beyaz"] = (df["Beyaz_a"] > 0).astype(int)
    df["is_gri"] = (df["Gri_a"] > 0).astype(int)
    df["color_conflict"] = ((df["is_beyaz"] == 1) & (df["is_gri"] == 1)).astype(int)

    # log1p of fillers and totals
    df["log1p_Cal_a"] = np.log1p(df["Cal_a"].fillna(0).clip(lower=0))
    df["log1p_Talc_a"] = np.log1p(df["Talc_a"].fillna(0).clip(lower=0))
    df["log1p_filler_sum"] = np.log1p(df["filler_sum"].fillna(0).clip(lower=0))
    df["log1p_comp_sum"] = np.log1p(df["comp_sum"].fillna(0).clip(lower=0))

    # transforms for each stage-1 pred
    for c in stage1_pred_cols:
        df[f"log_{c}"] = np.log1p(df[c].clip(lower=0))
        df[f"sqr_{c}"] = df[c] ** 2

    # pairwise interactions among stage-1 preds (this creates the missing *_x_*)
    for i in range(len(stage1_pred_cols)):
        for j in range(i + 1, len(stage1_pred_cols)):
            c1, c2 = stage1_pred_cols[i], stage1_pred_cols[j]
            df[f"{c1}_x_{c2}"] = df[c1] * df[c2]

    # ratios over density (this creates *_over_density)
    for c in stage1_pred_cols:
        if c != "Density_a__pred":
            df[f"{c}_over_density"] = df[c] / (df["Density_a__pred"] + eps)

    return df

# -----------------------------
# Align features to what the stage-2 model expects
# (prevents "missing feature" errors)
# -----------------------------
def align_to_model_features(model, X: pd.DataFrame) -> pd.DataFrame:
    feat = getattr(model, "feature_names_in_", None)
    if feat is None:
        # fallback: model doesn't store names; assume current X order is fine
        return X
    feat = list(feat)
    X_aligned = X.copy()
    # add any missing columns as 0
    missing = [c for c in feat if c not in X_aligned.columns]
    for c in missing:
        X_aligned[c] = 0.0
    # drop extra columns and order exactly
    X_aligned = X_aligned[feat]
    return X_aligned

# -----------------------------
# Main prediction: 9 outputs
# -----------------------------
def predict_all_9():
    # 1) read base inputs
    X1 = ask_inputs()

    # 2) stage-1 predictions
    X1_num = X1.apply(pd.to_numeric, errors="coerce")
    for col in stage1_pred_cols:
        X1[col] = float(models[col].predict(X1_num)[0])

    # 3) stage-2 engineered features
    X2 = build_stage2_features(X1)

    # 4) predict final 3 with proper feature alignment
    final_preds = {}
    for tgt in stage2_targets:
        m = models[tgt]
        X2_aligned = align_to_model_features(m, X2)
        final_preds[tgt] = float(m.predict(X2_aligned)[0])

    # 5) collect all 9 predictions
    out = {col: float(X1[col].iloc[0]) for col in stage1_pred_cols}
    out.update(final_preds)

    out_df = pd.DataFrame([out])
    print("\n==================== PREDICTIONS ====================")
    print(out_df.T.rename(columns={0: "pred"}))
    return out_df

# Usage:
# preds_df = predict_all_9()


In [5]:
preds_df = predict_all_9()


Enter the 22 input values (numbers). For Beyaz_a/Gri_a use 0 or 1.

9711900_a: 0
9710700_a: 25
9710702_a: 0
9710190_a: 0
9710703_a: 0
9712055_a: 0
9710293_a: 0
9710278_a: 0
9710709_a: 0
9710475_a: 20
9710275_a: 0
Cal_a: 43
Talc_a: 0
9801084_a: 0
9710146_a: 0
PEA0330648_a: 0
PP070G2M_a: 0
BE961MO_a: 10
9710156_a: 0
9710169_a: 2
Beyaz_a: 0
Gri_a: 0

==================== PREDICTIONS ====================
                                        pred
Density_a__pred                     1.255436
Tensile Strength (Mpa)_a__pred     29.064977
Elongation at Yield (%)_a__pred     3.101773
Break Strength (Mpa)_a__pred       26.505930
Izod İmpact_a__pred                 2.734687
Flexural Strength (Mpa)_a__pred    43.733297
Elastic Modulus (Mpa)_a          2784.317471
Elongation at Break (%)_a           9.020409
Flexural Modulus (Mpa)_a         2517.256452
